# TextSplitter 核心接口与参数

TextSplitter 的常用入口形成下面的调用关系：

```text
split_documents(documents)
    → create_documents(texts, metadatas)
        → split_text(text)
```

实际项目通常已有 `Document`，因此最常使用 `split_documents()`。原始 metadata 会复制到每个子文档。

## 通用参数

- `chunk_size`：目标块的最大长度，单位由 `length_function` 决定。
- `chunk_overlap`：相邻块重复保留的长度，用于降低边界信息丢失。
- `length_function`：长度计算函数，默认 `len`，即字符数。
- `keep_separator`：是否保留分隔符，以及把它放在前一块还是后一块。
- `add_start_index`：在 metadata 中记录 Chunk 在原文中的起始字符位置。
- `strip_whitespace`：是否清理 Chunk 两端空白。

`chunk_size` 不是天然的 Token 数。只有使用 Tokenizer 作为长度函数时，单位才是 Token。

In [2]:
from langchain_core.documents import Document
from langchain_text_splitters import CharacterTextSplitter

source_document = Document(
    page_content="第一段介绍RAG。\n\n第二段介绍文档切分。\n\n第三段介绍向量检索。",
    metadata={"source": "demo.txt", "category": "tutorial"},
)

splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=20,
    chunk_overlap=0,
    add_start_index=True,
)
chunks = splitter.split_documents([source_document])

for chunk in chunks:
    print(type(chunk))
    print(chunk)


<class 'langchain_core.documents.base.Document'>
page_content='第一段介绍RAG。' metadata={'source': 'demo.txt', 'category': 'tutorial', 'start_index': 0}
<class 'langchain_core.documents.base.Document'>
page_content='第二段介绍文档切分。' metadata={'source': 'demo.txt', 'category': 'tutorial', 'start_index': 11}
<class 'langchain_core.documents.base.Document'>
page_content='第三段介绍向量检索。' metadata={'source': 'demo.txt', 'category': 'tutorial', 'start_index': 23}


## 三种入口如何选择

| 方法 | 输入 | 输出 | 使用场景 |
| --- | --- | --- | --- |
| `split_text()` | 单个字符串 | `list[str]` | 只关心文本 |
| `create_documents()` | 文本和 metadata 列表 | `list[Document]` | 从原始字符串构造文档 |
| `split_documents()` | `Iterable[Document]` | `list[Document]` | RAG 流程中保留原 metadata |